# Group C Bark and Cross-Model TTS Dry Run
This notebook runs a smoke test for the HW14 Group C Bark and cross-model TTS workflow. It is intended for VS Code plus the Colab extension on an A100 or H100 before the full notebook is executed.

Expected dry-run outputs:
- 2 Group C baseline rows
- 1 Experiment 7A prompt-style Bark row
- 1 Experiment 7B voice-condition Bark row
- 2 Experiment 7C export-audit rows for one selected Bark output
- 1 Experiment 8 round-trip comparison row using the best Group A ASR evaluator

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%pip install -q transformers datasets accelerate genaibook jiwer scipy soundfile numpy pandas matplotlib sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 75.6 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 56.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.7 MB/s eta 0:00:00a 0:00:01


In [3]:
from pathlib import Path

import os

import sys


DRIVE_ROOT = Path('/content/drive/MyDrive')

CANDIDATE_ROOTS = [
    DRIVE_ROOT / 'kamp_hw14',
    DRIVE_ROOT / 'Visual_Studio__UC_Spring_26' / 'GEN_AI' / 'HW1' / 'kamp_hw14',
    DRIVE_ROOT / 'GEN_AI' / 'HW1' / 'kamp_hw14',
]


def find_project_root():
    checked_paths = []
    for candidate in CANDIDATE_ROOTS:
        checked_paths.append(candidate)
        if (candidate / 'hw14_scripts' / 'kamp_hw14.py').exists():
            return candidate

    discovered_roots = []
    for match in DRIVE_ROOT.rglob('kamp_hw14.py'):
        if match.parent.name != 'hw14_scripts':
            continue
        candidate = match.parent.parent
        discovered_roots.append(candidate)
        if (candidate / 'hw14_scripts' / 'kamp_hw14.py').exists():
            return candidate

    checked_display = '\n'.join(f' - {path}' for path in checked_paths)
    discovered_display = '\n'.join(f' - {path}' for path in discovered_roots[:10]) or ' - none found'
    raise FileNotFoundError(
        'Could not locate kamp_hw14 on Google Drive.\n'
        'Upload the project to /content/drive/MyDrive/kamp_hw14\n'
        'or keep any Drive copy that contains hw14_scripts/kamp_hw14.py.\n'
        f'Checked:\n{checked_display}\n'
        f'Discovered matches:\n{discovered_display}'
    )


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'hw14_scripts'))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Module exists: {(PROJECT_ROOT / "hw14_scripts" / "kamp_hw14.py").exists()}')

PROJECT_ROOT = /content/drive/MyDrive/kamp_hw14
Module exists: True


In [12]:
from pathlib import Path
import base64
import gzip

SYNC_PAYLOADS = {
    'hw14_scripts/kamp_hw14.py': 'H4sIAOeP0GkC/+09a3MbOY7f9St69WWkK0WxHSfxqU5XpyTK4/Kwy/bszJVO1dWWWnafZbVO3Yrj8fm/H8EnwEerZTuT7NZs1U6sJgGSIAiAIAjOVvlVFMezdblepXEcZVfLfFVGyWKRl0mZ5Yui0ZhBndl6MSnzfF6oKvPVOp4kk4tUlC+T8mKenanSI/ZTFKySxZT9I78f81+ipLxZZotzVTJY3DTkn5Piq/rzf4p8IXtwcb27HyeLZH5TZEW8LjPTl2U2uYzP0qKMk2IVp1+T+Top8xWCmyZlQmFevx++/nh0+OHLaXw0OH3fiYa/Hw2PP3wefjk9id98OO5ER8eH/zl8fRofHx6edqJ5nkxjNtzJ5TLPFmUnWuTXcVbknahIvqZxsp5m6m9ci38o02+sa6symyWTUg2N9yv9tkxX2VW6KOPVerFIV1FSROKvRmNwchx/Pnwz/HQS9aPbRsT+1zxP9naS+PoiKxhgvMyW6TxbpM1e1MyX6SLJnsqiJ8VVMp83OwbqLL5Ovu59TSd78aScAATrTXqW55dPVcGTs6RIn/zri50LDDjRzU2zVTop6zSWxux7Orkon8OUAMRVNlnlRT4rn5KSTuOu0ThlJPcNdGawlGURwAIlHS/M13yST9NQ6xfZLDtPFhj2PL66KlRTmjrs2xP27Um6OMeVL+KzZHUJNYv1In8KP1w6XFZUgoEPfz+NXw2+fDTjvkjn8zye5ucA9B5+dKKrm4h9iLIimqzLtKsamM+Tq4T39fQiXaVRwv4vvkWsBfYzXy+muvZZsrhk6y1epf+7ZksFwD5EizSdRmUelWyZFjPGflf5Ir2J+MJhrMvqF1DMORoWa5lPkxuNMplMWBNlPMtZN6/XS8B5NE8ZE0Wr9GuWXkflBfw5YfwdMdbPvmblTZQvYEASlomaaTRPy+gqjS7ZkoquL1IGxNbB4oY1m6zOU7YgWJcZ60WinSfrZVeQ783w7eDXT6fxyXD4BlhntL/XiXb3nnWi58+fjxuNk/eD4+GbeHByMhSLmtWxlnn0lE0NayadMnYsUjb1jc+DLx/eDk80hIuFwVwli2zGyMjq60nkgiQEwGUAzEGT/OqChGs2Ph1+eRcPfn3z4bASyTxnE8hFjZC9TfqxyxZys/H2w+8M8OTXVwxSYaNjgv5ni2nBBNAs+8aGXqzP2Ni7TPA2G8fDk8Hno08fWIdq4lilRXK1nANzYUSNd8eHvx5BZbysGVMu40RKBc9k4AodDHMWF+VqPQFFNZVrNAju1CWYJnxRxiARKrGQepzhxIiGvw9f/3r64fBL/HlwFBqa+MgLjJCH5kbwezcGUQuim2HmNfbi4oKphVm+uhJj51+fcY1WpIsiE6tHft+PYdZ5ZSmDm+MObXFSsrHzNVaQ7khhIFqPV/k1FD/v0HLeIVn2ry/cwmeqcHd33y3dV6UI8g517ysjxyybcAMjLiarbMnFEdeI4mfx1FNHfryJJaW7yxs5sXebOGXb2XhONY/4+EKrhwfRes8l2HMPwVTZC03qve9JzbMQNa3Vsi0lXwoNKH4cCDTfhYYvddmOW3igmfl70nBCaKjlxYBph5Mhk6jD+P1w8GYISmXUsEioRJRsXP4qk+IyZoZyarS4JMQVM23UR7BumdiNuRhOYyaW02+qbJUytZ4uJsIKVV+XbGlknOL4K+Ccx4vkSmPOFst1Sep8zTOGa5IvphkQglbUemCVlBoHmMBTf1G+LgFMaDTYQlgFXE16vjPLDfYruIQZzSUjJJOVE0089rsoWbvqA9vUAF82xmZmmPDf23pWrvPV5YxZIoRUPuKHCOolhqDCdL2SHGYGQmeLAV0HShZMISTz7I906p9+DGqVuKDXqdbAE/NnttBgFeSmFH5Wh8Lrs/jR6I4sEkzhaQobgniZz7PJzdbEnTDLdnFfwnsqJNcxIjH8RGTmjeFy8eGBE7G/NatPLtaLy3jO9j7lRVwYKVRO2E82HKdNIpgEt/tYerG+Yj/PhfJQOPIymcdskqfWJ9gHsO1DutLfr4AYEt6Ln29o+EiI9EKfi+wqmycrZlPFZa7V1r2pe/L+8Pj07eHx5/i3w+OPbz8d/naiqdxqvhvs7QxACQb27+0OqvhKVbS27KTSa1XJ2Z6TakNVzdqRt0nnkc3/5/R+624dD06HvEsHOzs7bJP3gv+zt8/+GbMd2++47ptfWW0wz0+Grw+/8J3h3k53x+CF3RafKij+AFXNaG9tlgfjmHB8LzqQpkOduvvhurs7D6pc0YvdrbrhHZ+eBnDOuPbLPCvKVsC8aSvQ1yBzXj6S8F+u8iu2bLF4Cdgi4jOT0Mwm0rhSI3ldo0aaFdieEKpbKpeAOeMW8A+kj2wNpHyf5pNS5+kilV9tMcOdN3HJKBAboYUGicqRlvAKJz4NB3Wm4aYo06s405Qq8vWKkdKySUGflcn5VuQMUu2+9CGe3ocRDX3FGjhflKt8Pk/OsjkoCTypq3RdsOGAm8zDt4FJALn33qyVo+PDz0enbDpaAsr4+sC6AWffyXqRd6PBYtqJ1hfRf6/3dnb3ub/sA1t+l2m0zP74I+lGo3myPr8oxv+9aApMr9Ylq5LMizy6YNSNcu5QyxZMgYLHKirWkwvwMy/nyQ136mWTqEzY//O022y0aY8/mh6Do4v11++VbGqoV4Pjj2x0Q3AdSQ8dQH3de5ouQNonl0x3PG+aZgQl4pPT//o0RILY7M7w0me7MtbvTE2p5kjqLsUk9JDsQRQim+RwLxlfrDydrOSGmqgLxpjMtA20ECDD3zSjwNDPinzOZm1+E83zr4aVzpNiySpsoE+nikB/MwRCzP/3ww+vh359a0vxHmzUtPi2xHkv+pIvUq3MPLCi4nMPpMOCm9G82IzmRfPOlrUvHY/0mIpi7jI+HbzjVZDLv6Od+h3XYW/h+Pz5hDfDUDx//txTdjQcfFSmE6u0yywgUomvU4lhf88aAPiioYvH74aneLIsTc2o8ZJbhv4lKvUuMz8YGfgGV2gOXUPPQE28guUr0PIKVVhfmRkNs1wQ/SKXle7TBOLMMFlkHcFTjWk6i+JrtklhqIqv3InVmmXzFFRrj5+ydqJZls6nsNKLHjfLRkw8jDsR1JUfYKcKXztwyjoet6Mn/85he3wMCl93mbAdT9m9umQbiZb4UfRPV+u0E6XfGJ44v+Q/2xzsOisvDCwcCLaabOscLdJr2BT0m+xvtoHKp4yH+811OXty0GyDzLhg8nie9rTc4sNbMR5jI+y+YV39jX9oiXp4fH3zZ9sC7/J/LtJkyiD9hZx48B9RvErL9WqhR6CIvWIoYnW64yM4p56PrGJE2Sxa5KUhDKdc0Wqb8cqGR+MgGVf3IaNCC72CYbYjZlYBFzC5rUl7LAgkYNuaw/jhNngg+JFOLP2JLT5UNjbJJ3AoKItMGABAyo8NTFdc0BIn2t3PH9huLH4zOB2AbfBl8HnY4Sqq7y1nyuLth3dM1yyZ/UWrnLCN3mlb9Z7JiP9hG182dXNmKzIjnM0Jo1YLKBqD2u9F7Ev0f3z62D+gQfjQ2FcyaQbApmpTmFZQgTEq4DHYBTuVqxsHiuHn1bq6Z2XewmEFbQGbfpukyzL6O7Nm0+Fqla+CmPSYXWchkzRFS3xOVqvkpgczBwEIyBHWAx3Ohz5j81P28IRxK7jF9oUYSTt6Kqq2CJ6IcdZuuxO90P2Rlj2TYi3skOMNhkiOK4KVApVcystabNO6yK+liavWriy8ZY2QZtu9nWfTO2ZiNv5Dx6m0rpJvsL/t77YR0y/EvIhtWwXzyxaDK4WsJFGWfhM9YsuQ43cJE5wdKlk0wfQwFUQ76vcj8IgYsskeMS7dOEDBe/Mi9UHL9YYHpI4ZODmnLbc3RLByOSRBRs4EjTXF1LHpKs9LTivQ9S34D1+2nUjuJqHcs5BZVzVzGZ3GiIXAgLlgfVsMBiVyMaPKiiwMBXQCKNycrm4gPqfpAJsj7hE5/x3DKbKC8tB5E3hDVaqrjfUizkubsAy+aIkfgoZaixk2g0+Sz4QXMV9laaHPtuU2OQf7xmCiB3i7pAzGb58Hkup7vurOITgBeeYD8ZyQE6B9H5Dn+BxBzdbzuQPFP8qdjVCHTAZyfZAtMM264JNIicLnKmC7eUQI1XSul1NYw/KkUYeYCe+B+Aq6VCwa/tGsIVTHsID4ejbPJ5c25DQtk2zOzEcqiDoNa5mZboBLkEbItawIuzbWtFmRLYoyWUzSFo6Wg+YQ4Qj627uGGQawpintMhHDaJSs52VLrCPYQN3etdG4WWslCAtRjCEM7QwMkI6CiF8YDuoYiEkO0g0OhDhFC+UjNZi650zoNu16rKOjcVsTx0yHoNPCQYzIY5V0kyUzH6ctg6PdQHNcrK+uErmsuXC2u2ZVEyRs+zCMTBNjhU3yjGwS4RaM29Ldto72DVs2OQciUdW2DvIFpmmcQFUZf9myKzkE7jmEsiDscffs4aKYAf6Xw1cjGC5Qwgwb8erIcNiYspO1iGqRqlCiyR7GPGHChJdo4cWZ2lNLbqc5B3uK+fA1GSgSTAFlnNKgV7KiHSFgKScc2NFie1+2sBypEwk/LueOoHkk++INT5GOgYAy0uEe4g/sUNPRFz3ucMZQNBCjF3GJboq9IRm9SIxQLDd6cM22GIiT7bANCuk71LfgrQAPCu878rbgUeQChcUhDRYMOghg9MLE8jhDmg4gPbKnrXrDJpSNqsJA5RHg8A13eeGe+UJQrD64oSj+CiYkpRfeeeKuO3Bt2D15akAkqKzRdtvFIS/btYwhCWYcMmOT2z39tibbHHJQUHT4YUQ0BhRBOBRIBuaYJu5sOaECEuvIiE7k7o57Yhv7EPFRYZyGRQgNWaGjtsJZLALfV3Lcf+VuvwApFk8AU88zFUEpxQXR1pIqHIlTiQoH5FTKXk+vasjfcPhPJaqKXl3zmwyEf9KVU2vi1Jp4annW958iAHzL+lnyZ+j+ym2i68N/NvhZl7cd2mazk6Mj6yxi3AANkqPorQA6GZVWtRwfY0mLRn4+wfAIi1oFAW5e2Co+cPPiNqGDNcSFjir8BxMaZxVC48+UE6+8ckJF6P28guKBBrUtJDh7fi1IjOwPFAeyG/80AkHHCtcSCjqUuKZgQLHGdcWDCUauIyTIPuqxRMB+pdngRh5v2gFYEcNRnzuj6rCqcIaYSRUhzJsR2GzQ7vKjzla7rrCqdmNTafQGFzkxoGRyrZBva4ZJpCiGw0HhIbGvAznE6GJdIFziTXCB0oVjVWO7ZzhJ8SF4m7AC161g7Ufcj3hviqPSLVkrg835DSV3gCh0vRfBIarNDm5tFNVurQTDiXIWoCrMBCC2+JTYbuGweGsrGA6ftwdmxdH37HXirxsIrrdIGojA/9nsDtYvcLW4CQZa+pqkP9BGn12K6JHKs0ufSBINw9RAGzoKy+MZViYEDC+/Hlkmxdhy8koVwCvSm6N4DwnFE08xchtBzIGq6pmFse02VudmMipFk48XikNP4XS/YYMNZHVoUarIYCQ8Xil5JSYx0ShU2Ihsjtw0eBs218hvfMCo0YbrSCtNCI5eRLpFypyVLOaJAOjvTuVJoPLEV9mdQxfQrkOMUjMH63nqjIqWEkARUb6Vc41jCxz73FnH/eJE1nfc72biaCGYDmGGTpQU3B/rnsziatQqmfyzhDCQG7/3jmSgWB4voGHykwY0vPRVV7efSc0DX018Nfqnj0WY5CmsYhEZxhuR0WbybFza2ozcO90dO97MFymHMJGQuNbpzVKExHVQeFw7AC9bb1vdhDNJbydFYIHoIPs73D3AgLt4rz4CEruHzBbJ55A+SIokLos2By6i3zUDGKHqphBGhKSbFbG6I2CzD0CjGEbGkDx+FjXmxNOq2OU4m8UyINYE1ZohbgitxV1X4G7PawXc/shg2xkTX6UgQ6bTufB5Z/DuTnbCwDMe+nOZ3jBDs1wzi0FU6Ha7Yz8/QG9Ydd4bCm9PZYAFWVe4LmcwbRNOzehsK1h+uyZAeJslbH4ksaxXaZnwNF38VhXvhCFNaCH4rObqgRmcmnXUkPhwGOE2s9Jt4EQZ7mj6L7T1QBIKYY3Ca60AUDvskqFqmZ66DgF/J9y4TOyHDPSuRkBxx4PZ3irxbuvYDtd2D04arwv1Yjl9lZGVXAHCH24TMqESkgoUtW9Sk6xIo2Nh7nLh3SIsPGu+gw5EA44qGpwcR7o54JurrCgYRXrRLW3qrhu9lpFH0YfPR5+ePGMIGPFTPstwbUrgfd1t6vbQ3Qqr30Za1ZJQy+QGJoNREgjShb+VJKIik1fzOaKrKTSjtRURQOLYZGi2qXLgLVKjjlk9W4cCdSIrrER+5cEEPzhYyLLi7hcsRNymTkBQs1kV7kNKHxzM454ciOu+Fowb5mN92RhyQLrtF2/0AMSTWsbqlC+wZ5vwGQToi8v5Kx7okeKB+L9ed/8kVumdWnL1emWDdB/hIzoU0owFhzATzM1CYV30Nff55Aoq5jdl+HVxf0/otl4GTd9TDLkbSOcQkn6oJavorUr0656L2bohi+efJmOwzzZT56hMJGp4BGGEkxA8zop3U0JYUWwScaU4chMhbC/SrEwTLgKchsIOIwoYgCTMw7ICHSS+vAxbS4xgsoaes8Kqz9/4oZEFIU+Jmv72lAPc14o+B62J9hHOMyexdIvCHkjsfwSmGvIOZ+hA0s2XvQIV+xNYYEEXFFpqnwZbrODObdOqCm1oSL6RHh3dA+UBNQACu976wsBvnGCX06ZV3e5oWo68Fca1Nm7EF1e5ikmDfnzjzas83KBnzbMmwQOIj559aVV6Xn4l1q834UovwMhhM+LAmBFozfnX1iY9L1egSpqDlpdKkYM+kU2q/P4QwyDgLzYpfHqmZ52NKwyIIDmVpvyxrTyV+6enxxhalg7mh+tqB2WNxamVrDvSGpo2tOocZIE0RrX0poPMm/bIFmck/1GP8tf30LIP0KR+0EkFqDd8yS85HOr5Ezc5kVsBgeKgC+V6qtr/eMdUyyDgDjSagJh3oEVEikdwEPcvRzApvnr9aE565LE8grKSHnu/itTe1jl+0Mmv+oFcSdyoupYUUdKkHf2trwVKJBDLRSqz7Jh9CK8LOXMgBWF6dZZOwRfWpF5n4IFssU71RyMpwG6pdMF3IrScRQ4zb6ZaKraolxw15yQ28PavllVl+669qsIr5nkI2NumFclBzShFciKndXL7kZqesX3hkhhW5odVq4a9pOfcyO2Yg0kXiSl/oDG1jUGlGvVX39Kkqm9WqWZ9lR2jqko8uivFDucJSUJfHnQrnsdzmKMEmEiWfj/Z9WJ72WVys9sfjMSSQVRV0oq3LC90q3MzFdwlDt70PXXslebRBnC3V54xItnIo8O2kH4b5EQ2C6wZcJswxniyy7F5M4U9rogkSR4DApKK0QeKSOrngrMyPXgrfSioZgMGGgSdsPGgXKJerLswXbA2ly3SWbttpoFevGrys0+KH44/mruAabe7I//Zad7RoS1XWQ5hjxB8oAt4uI6vnUEzCL2LoItN9DMMrRIFtBQqni2sg6akLa/xy1MitBa8gQX8Y+xgAWrnK2ZKtgyCDpxL9+fJ1dk0iTK2P+nx/452xm32/8a9deFmZwLXge+aQTfCPfTfZt33U+m9H6LzvoO+C+o6kILx8+fPYxlngzPNblZ2Rof4A4CVG0El7FuxHhewhiAWkGmVCcR5hyKBtQNuQ5CLinb04zEqRjQP6nlSRETgW8kbR73dceSLBBS+zDCg1mSiKRBtuNEejejlo4c6Nj2IUOLh/fDdK7iFIuRtjOyyMUXklXC+FrheJHjhyxbYQv3htHw5EEoAN4kPV0jLuODROvDK7YB9ckM6YRfW7YjiS6U7bkneSoXdfGMs/y//Inp11yaYzpj9dUnyh0jUcrmdrbP5VD5MJRKPiVelhNdOp2Z0woL6kfPylXJ2a5FYBe1/+Eo6+VJYkSJekLw6xnRWR2bR/Kber1EV9/ad9kn5rnyoRq0qJxeciQfy38jrqCU6YfXP8nwONIBbMFXue0wBFU1HPrYriaZAQuXGcCCfddiN5FY/MK3F82jywW3Y7UpekZhkGiHaK2fT43RAggXHZe9nxdElD4lif9rFokvy0SFfUtG63atAUaOrMmeWnRTQl8AQ1eOXuCCBVbpQGfx00BDi8+jfCYAdLmRiYVuz5rFIYpxOo1uE4U6gk5kgC5FiNNHdzZlBDD25xc3cdZu6M/bKYj3C6zDcoaYDOeFvnvKYXtYhhAWaEyticQ5Sgr9g2tLz3iZaG2J3mHwFjcyqd8W4WqtkcZ628CDaIpgONSMRmQy35LoPulKBXjLA10uYnLWu/7nv4oQfhBKRGN7za+dGceAxJxHJ5Jwcu1FO6PEe8p0BzLJzMgS4Dik/jBtIzIavXNE9OCdKx0o0uojSxfoqFSk5rXlDjkqZvhPSmwWTWFp752wxy1l9CSm0Yd40tWzqyCVmgBkIRFoy+1nnXcWFtmVvLDBNFaWeiTy5dVS8zU9BQeayWE9SNVhNha855PKAOFFtmnbp4pzN74V1aDH24PCyrWi+inje3vgZXCDzqWkfFmcNyAMBeYPALuZpdH1o8HLpReEUymFQuaICwDK/smce+bKzoHjKZVr5DkWtqhtH59kZbGGp/iKiDPxiyD9mWJdrF7UbHLlkZIvi3/qyxoYXf9roQiHDCoss1DeGNJKn9UQhtCu0B1XWpxepVGOcUk929yNhsDL7SxgxYFAnXO7k6/MLNoq9nUiMKprMs2Uhdln8qVzVCWaLrKc33ahJ2zpO2aQELWQRN8z0Zzbjy6rkdiv4EIg+s6fNna0QsUY9P60ciasueGpBa1MdXyC7HlEJA0Kc/yX9Slb+e2Lr4PTwHcRMbS9oyGKiWOy+NgIOpq3MzoeZnBXmJjU1zY9AKhBRzfpC/R1si1nAZTD9SHFL/+XbN9H3jzu+/QhN/U2xqQ0F/apNPPq57gYByOpD6LkESqshj1/FJVAfekE7nhs7Ru8081elhcQIXQbvR9bzz3RbKMSf2hC+8Gwa0++7ZcRXa+llUzJT6NvGaRIPjE+y5U03y9VbAtfJV7jW1UBXdpGqZdMBthHrgazXhS2R25PwNlG6aWHyMJRzUImjRqou0/hud0N9btPx7PnOGR3PnO5cAJC7NMM16CKQdIhSngLkRgtRRuljW0WZGE5dM8i+bdukeJxtdGUJUtbytCawiPinlulUJwrhicg163lyls777kvmTkMjRa2xYlG8AlFFufLSRQFh7eQx91ZIFlV5RmCIQR8UR9jn/237hZkrPR0YNHhF04DccEBraCE9jpFdNt6ojhCsr8LYDgXjD9r3LCKgSmZAkBbFGvZIrMixq+ZQN4S7NqTunLHy7+NK3ecfIwa8s6/nmuvXLTvT+ne72+8Rt+Zu/7ZXuX03MVknzGBUtnFz5F7pzvmVLbblkp/tCSS96NYgUze+0Pk9SoG95fVzPxLz9ksKiOQMLefJwp2heyVi8OZjwbIFzOSgyBHJiNr3TJKPbvv3NzAhGRKGFns/fTI0/H34+le+Rfo8OCKE9EsVUyOQTLzpzSHelPMRztHe1MGdJuM7eRKn09gyNXyT0N99O94103H9KiFZZbdTHBuEpSMwLXBTZAMRAWpBobKx/1l7WA8LnAmfP0mAGWSEIosLem4LBdw3xm0bgOa33V1oXIlg+MoQz7KJ3Marc3iKwFdnTFKVqbUCUeIms7x7gax5UqbLaI9Jsdkq0WFCkeLIbgTlT+DsEhrk982YXXBZsF3zfM4EUArPWMI2nAk7IFwB8nCewNtkGZgzQCLeTcaK6bLoWqfEzGCL0UXlFrlYps5cHyqB1DmulNa3CLG4FH1XW2yrO82in7fwz51ydd9P0FGxFX7XxyOzaI6W0NM1bdsHrKO56MGIcw7YDvg5fMD+Y8A2fdxNO6Grr0i/5W4hTWnZDrozDr6Z9GpZ3rAZcHp91yStWt3f0PSxcSXVbN8/cM0T9v6jg7+EshdYG1vH/MVOPhPu42Qh2Ni2eugEXSHfeECwIXkbqfisbsX9jRVpugN5HaTqAmnDTUujx6nCP0ZEa5gX20WclXblchnF33BXGJRpyVZaHy3CkUiANLZ3sOaJ9wDms4dhfh3GPLk3ZtfDbSfEVP8LNA3abbUSek53wgFGrpJAzzpeGMTXfYfRHQhrcF4qDsNUTO9DxTEJ9ZG38QwTziC3kcWTNKqErEt1ROV/AwZhZWajbKxNI0jkmCALrVArc+FEKFDQpu369QzRTenFY4U7UeBd+w4dB2qr8m0uy5xNrPx26rUk+sWXlEwnJvOmphDP0co9LrigaGchWTsbnLCj6xOjjczJdgMdjjLZSfQmE0JGLW6IOTMVKcJtXh+sTlXtZViasERvdgfxyfvD49O3h8ef498Ojz++/XT42wnlX8CCR6uTVxISuJFU5OC5zz1kGGJEj0zHbQcBOoQG2xtTaYRhPaDywnIfSwCSY9EEgfllGSFXP5DdMiTI7DNwfbLtByUuVu6RdBFsPsAVw8Ynyn1Kbuu4uaozkrB9/MNfneRC7zftFOhUv4Dzc9a8JdS8E6/b3IYeUW3fedD5eEXaM94YAC2rnBd1vNUMB3WC5fa1bKepgDUX5o2o3swHpr69gUhVaqO23tjTotJKWWrpjuHvR3tab+ip+WE6I5xf1dEbprOVOqOaEGG98SxxFIdzylypPazaFm61skxQ9ABHCmgBXyss2g9aT7lY3XEVirov4mkKdNiJq04CasimaG+jVLuXLqL6qPZLx51IbNDCEobqKNBMJMCgUkndS1E9UFlxQ3+VnWeLZB4/nuYKhQJzpIFgo0fUevfQfBtUGuPLW933u4s/ttRwfpFONuTOVsJ+T+oeG4jamuCZFoDOozGuLniGdcGzH6sLnO7Gz5INCuFZDYWwiR4VKuHssfcSDON32kz45O9Zlfx9kNx17X8zsqoNgE+wCieKtFbhSdxklRU+uXpP0fg4kvCBkmwLCaYkl6CLfFQKns4BA54tnvrSyp6zKumEH65CUukviWRJpLMfJ5H0ozra9sDGJLi0ua8AIk0/wJFqXVvSA+l6dE28szoewq5gLIl07/g5ldtn6zowedVHCiJdeWS/+mPJE/O2jwuJ3v2xoPgKA2HeFF5U7h/nDd3S5no7e9O7mCO6NejEVxiZkB8bJZz9GlKV8bid15V2t09/utXNGPrmT7dahQd23yfchMTi//W1qTgI5KaHn3yy1vcIU5VsgxfqXcz+C/wBtt76YS131kehp4PgiGW3u2OL4/2gON4ntmLgYKq9hQje1yLHfZzLlcH7WAbv/1gZXPmYmE8C79eQwBvJQUXwVtoQC1zxTAYVWeSIDXas4Xecwr604KKFMCq6ankfxlVZyatWVr3Z3jDj/lmvmnmRntrpb6wuRnkauHVeR6Kkdl+fIqG3W9IMMwilmpvyYpspt1xyM9jMz/HLH5tYcEPMVGCWAmFTMvqkImzqr1ioB8ZC6SgmiyTkmMx/dObxpvr9pR7D1m+1eqSvX7j6w7rYavKMhB/f3euQzzfCe3t+fVR4gI3uo9VDFI6FzxUN95RTrj5ykwW7EW3Qkh0D6p90b0epxAsFpNlZs75PbJoUiL6Q3FD2LhGba8WD2b3HD4n9g0TVvX7EqDp33rX3veqhFRGQpPW5fBfRrelo60Ywlm/ygFi+iRvL52rs2Ph3ZPR77UVoQNEDMJ7bXpUNdjz0sW+B3SvO7GXdigf1MdbKO4RCzmT2GY9P43V8dHz4+eg0Pjn9r0/DugmDCAxuAW5sOCmJRLKULVIRcQDfaww83E1V/fvhh9dD7InZHddpwQZD7Tx27/mkghCEbvOEczydIoiWi3Q+ZzvN/LzZrkJsYvtaJqOzzhdohbqoCiDHaGs6txWYPEyeyicZKLzOwarPJWUOQX4r8nTw7gQnYyAxieDiuUCxXB7/jGyX8iHd4HDnhWoZYtnemygswXFUm1YHsYGsoP2ylLG8jVZZhz6o0DcGI5PbU3CmT+PdF/iJjnajMu6t4QnHcN4/cjdwPvJ6/N/cx9LnYYDvPZs3azH1gVAMVwEZFMTMeIB4hH2/+Uo2/iT933XG5CNcMDfg0Ss2u/AbrnJy5lp9ZQoxKSOg0xNOqGj3RXT5/o9ucCeMtvcw4Mta/JTKXAe1+Omj4SfgadoR/FiJhno1OP7IeG8IEdHyhOg+PHhZgwcvf34evNyOBz/W4MHahN7Ai5wBxeQ9Ih+qXZXOjPrgwNLT05PvE1pKntz9sRGmhGrhk18wjFSudEZI2+Ljz+pycupqTQoutUkl7MtEVSvKmzkbEkUh2LEaw1lsWyEWEpH4rhoJP2RkXCkqNxtE89pqUaRStEw2N4iIJ/gwVbn10vMfHOiDmAv+nlYS39r4R7+gFIW/jO84ulv4z12zVnxpDQVPBLPTgWZVhAqSs9bcV0VbBo5LVFa/PvwnFAdKha5INFQrFlOzK6WPYXa1x/ATh+7b+vRnrbjX7ShrMtf21QmLm8+5duiuPqSpkYY9NLdELGwIpuV/3MW3lEx3Bkfd8NmXtcNn7Sfp7hU+S/J59iGX6KZY27BdZ5JD6wXs4QGSgbQimsx6v6KvP9wv8Da0Z62k9G2we+4TeNW0a9IU0FXk87+Mtz0FPa/nNSuahUypYWa585CXaJ8zvuZxKp4R2lzS2ztWx0B32KotrGXQFnmTkrmMRc9uLeyjX6wPD9AzVYY/0TOGSH4KE7vf7jB9wrCmbtJGxV+qyVZNmybjL2X0MyijVzWV0ceKevZGL7CyUDLqfyqd9Kq+TvpYXydV6ZHNj7ZuRfMHqCWSIx+dxPtz6PsmQpwNWe5JnYmewFNdJIqMNrqVycl/EST/Zdyd59fpqtXmQepsV6Yr6OzlWidpUPaDfWx6GhIkKjwKCu/2XH7islDlTUdvTowr48ToZjMUIoZIEHK/SGUnOuB5qtcjGrlOwgDy0Rmn4jSXobN9SLzVwiC6CN7WgAMVDzhSABjU8yCnxz2EksHK/Gb4rQ2Mz5s3tt2JQvUDz2/s7e/s7PiG4XnKok9fvcDY6zye6UoxWBSYDc3SkGzZ+/n0fz2etxgBD+pB9gBF9GdZBWhBMtuA9OEXasuB6PnZjAfc3y1MiNebTQj9toWT+KdqY4vXDVaO1puMm8wRV/YZrbgB1WNYI9SBuIWH+aX2MIvX4YlzWR5AorjSlz+FS9l5yN4XUvrS70l+ucmTrNTtgXhwDudTcQ/Eb+8CL9d4j9W3fb4GzL77PVXjfbalKdTx8y1w0jeo7adw/E/E8aMX9424ul1Hopzj9vshauK2JpOgwAEllW+mU/0DPfXYM/iBsPBuxvfYVF/NCzcT9/eavuB631NT/eBaoALhYOPByEHwYIQEAnTImb6ORrD4Ws8fcSLpJ8/suTdhBhCuoKMMeiESwxzpqKWq121dOQmtISTBmwYhVxF+v7LOZWJuohh6VNwADp/6iqkxRHrqeSy76jbaTLD/rYK6q1Kk8oFceYq84bXP6nu11mxtvdBqOxvchfe2Yoj+BXivkYpHFgn3qriY2szre9n08flWdusfhWXPbZYV+2fvO6SV7Ax7TC/UhiWgjXwX+Gg4+KiyKvyDLId3Wy+HTU8fbrUmcIDYpnVhqWtuEHy/ZbGF53+rpXG/yJ8/aXFJ+iqTo3KRaUtuu1WmwaqSQ9Q8c/gpV9THrVfURgsvsPt2riX5R88v9KNP9qL0YakZfD78towOJL/1olvNeDoS/RF8QVv6gSwfUNWSRF4fRJ5Rs9rb43p6CHBdRw918hxscvJoB4+hsaMIa3tznDQCB757q5NYleARdsxcdnzGf8easLa93djC/3Cg/Q8HwjGgn1nv0F0lckIc/BROCNTfDZ6IA78nwtAJiuzbONZE29GY298xLNZX3BmxTG7A+gTnRfV1wwCl/rp1+M9w6/Bl7PeWedjcz8w/6jLhywpEXmemd0xhHGGBtPm2n7kCJJFX3hBqf++7fPTuUcOcsxpRUHExaqbFgLh5Fkuo6jtRGHXHljm+V7CsKiPJQ5DfR5RwRx2QA6OmL2hRFPLS31WSyfc7PMlP0EMe+PJf1a0+jsrdZPifh8IXtnvedP59fiLd8YE5Vxt7m+5fViAjUrxXdRXSRnJHRw57OTIsmwThUYYQuQMN46wYdwg9HXoYdYASjRpWMnmvhpvEDfCac00ax7wvcQycGMeyB/J9smR1vkxWhUzIzv8ETlSfu4PV+Rp8vEe8pDVN9bul/eaRuBQL+fazBWTILqP3v+3uR8YzLEkhdDJE5vE3KNRjx6K1bjJlykc202o+ecJhmNk1uYD9a9Ef0bQDUfjirY/e404kvQd9fl+2qmV54GYatu/MImRmrYpHvlbnEC4hEfN/ALXK3r1cgR0IIqc7XV8t4cFHJRX6UK3L/xTSQHwQiyCC5FyLsr8Htm0jjpmtBdMp7881ByfH8efDN8NPJ9IwasI9DOuLcn2rD+Std/Ux9DiYKve/5aVKfbd+VZm1HN3P7kTaNTwWYBMIyP4e/z8jM2w37u8AAA==',
    'hw14_scripts/hw14_experiment_runner.py': 'H4sIAOeP0GkC/+09XZPbNpLv+hVc3kPEHEceO473TlXaull7kk1VnE3Fk9yDSsWiSEjDiCIVkpqPzM1/v258EQABiiONnc2d/eARCaAB9DcaDXBVlVsvilb7Zl+RKPKy7a6sGi8uirKJm6ws6tFohXVW+yJpyjKvRZW82kdJnFwTVr6Nm2tRtE5S9nIHL/NsKd7/CI+soLnfZcVavL8o7kPvfbzDd6H3gfy2J0VCRrz017osRuKh2G93915ce8WOD+z69uXrKC7i/L7O6mjfZO0Qk3K72zckashdE9XZNsvjKmugL1FwS6ooIVXoFWW1jfPsd153VVbRljRVltRKJ2ncxEYHOYmL/S7alinJQ29Nmijep1kZ1YC8GsHeRlldhl4d3xBWxH/XO5I0Vbmu4m20ytaAfF5A+4+rJlvFSRN6TbYl1Wj0/rsf3n2I3l1cXXy4vIp+uHh/6c08/8cyv7/47sU2K9L65WvfqPX2nz988923WI8UZxc/i+IPP37/3RW+bao4K/zRu8tvLn7+/ir66fLDxfsfv798F/10cYXgX76Jzs/PR6OLDz9F7//57vL76Lt3H+D9w8iDf/46fnUeR7fXGUylinbZjuRZQfyp55c7UsTZC150VgNmcz9sWy2j2/jm1Q1JXkVJk2ALmCtZluXmhSg4W8Y1OfvPN+fXasNEdpdmFeBvSGepbJOXxRoIux3SihKIJNfN11FcV9himyVVWZer5oVWEo4eR6Orqw8uDK1aQE1TOwBhSWhtc1MmwFquAVxnwDpxobZdR9ttLbqSaIV3Z/DujBRrtfJ1tIyrDdas90X5Ah+6qNj0VIK5j/5LKoLxNr6rQYZmL4NRSlZehPLA2ESwxzjwzv6G8j6lHTBdUMVFjYQhlRQsUX9Eq1UEtFMhX47pSzrAeN+UoHmy5Iwh5QzYolwXGWouPgn8R+VzpjHy3MXAC6VZfBcV5DZqyg0p6tmr83NWFgyb9jJCPVMWpGhqNvFmv8vJnKo7+G9xAAn/DcLwCwjDN2X19uptKJ9/rMqE1HVZMezsxCNwXqfKBKFHO8AgSjtJxxYkGPK4CEYSZwpINooT4U3ITZyPA42qYqghqzIMt8mpuGVkx0mVRUr5Jc6/JQWpqNULRYU+XBs1hqCmo8FMZPcP6+Qung3/6b+sSFsU/uJJYktOZK0PdNhXXwMR+c/yCox6KAt6eKpTZQjFDWNlsJRjOCcCfjZGWh3A9lNRjnO7KtmLFuX/AEv5bVw8Ew00ez+32Xk3DdTxHQ+Y+wUKaD7DJ4IU/oWLtLx8ALVDUXcY2dcnCtkvWVO/Z/3izyu00NAPJ2cjHgFBWvEQ9LQulEFF2efxUCyIlGPtFZtXCv7QBVPRR5tFWTr16qY6ApcXoHE5LvFnj2hoxR0siHHoWJPQh9QfplYQE7j0qtjCi6GKs1GaJc0c8KDO3OKITmibiELh/Vn9tr56ycB66aBeyUBoq4H11gPqGZxkqcZXuEBHba2rl8592oSkUQzuBtTl616dkLwypx9YlAjDA2P8L8qzDaHs6/0PjRLAnx9gTJSiyjMjZ7byZCMvq5USpTd8yfgXm88olLavQBZNthvwk6CkQgzMrqo9rMTJXVY3Ubmhjzo3YgiDz6DO9+tsdT8Glt2TVvjgLxtNch2DkM28+YLJH4gQvvKywqNNJnl5SwDR7dhhZlhjktVxXuy3apEEOIl3sHhNx/gQyGJgrxXvMC5S9mt+9nLh/QVW+pHfAwdKtRn6/uTXMiso/DqYwGyyHa3kwfh9EZnwBRKakgshjW6MWfgjrqr4fooSSDFS7CZFyt7RnuhPZJPdJK7pg9ouMOvUv+0J+Z2MlVKYLH0CuNnWm82883aGFuhzVjlrCOB0YYBIm/sdmWwyQBtQ5sHP/NDz9/5jC7FO4pwARFDG43hZjwFylhWrcqwACCbbrAhCz1EW3wWdAbIacY1VEOYqL+Pmq1eB98KjP8esX0D7S9aY5DWZDgajEpXWkXyLcSaMqkXoWlfpuNw3u32D64WpKm2hl8dLklPODj1WdyridK2OFUxvymgL9bCQCraKuLQqbQEZ/oOUNTqi4HGCw/dp027UbKxBC6lJGLPhA4FA72DjrmjDFPSWgUAYqkrG5HW8BcsawWqMMPM1ZT7qKs7zZZxspsBCDYzfHlSjmIIKU6kPrsscXKaoiLcEmW/sr+VaD/RyscrWyIz8l6INWDvoCIYWNzBw7oMp8EKmQVXdwlsBNYqyocVUWQgYrBj6U2bpczC6AuEYg6nwRhOlyfFdAls9vVPRSKMlVhI0kWQEHqkobbk1plzDmb9VVlJ9w+izOivqJi4SwqvJMHXQ4WVkH1ZpArMb+wgciecHQQ84EewOKFrYW/oT0WWpP6Zit7xvSM3+MKXYN5z5+aLD56xEYqYinH52JR6CDsrWWRHnFNOUyUOviSsUDPmmV9X32IkWPUA0raMANTu+VbrqzpTpNunh1km2u5/U2RqgCA9Xzm9X5vfMrU2zm4w5teskHXe7Djsds3FSX8gs8l68EAAZ8PK24NWMCRkV+RS08THLEUJPIQUU2HQ7Jxw8phht4/hEraW6UcpioCUMnZm6JmAYy0qBLYC3ynIeqNFkLJT05HUmFYlTpXON0To0p9RmiDVE17rmeR0ok6SbHHQLpgas04qpDoXpXsVx1GNSvKmy6qG7MRQ4L+OcwR5gkmrZuLsFA8Ye9OzMtusSejUMrJkpWy6BAZz/msRpCqo+32+LsS+mB3Mld6A8cnBDx7BiW5NxTooxLw6CVqVoGFDVbCsm3S4TYCjZJ6WNHzJ86BiddSkVaBSWeFMIBcNNYZ3H6EXuKF+7CdVvKzu+xtRAYg9vUOg2DNECdKUsjGib3aS+3q9WMIkANKk2VcscCaMdV5F/5mnPZTU+p2DR+kPUSSE0whopqlwoxRqcLRib8IbenEtrofgJA3wl05pYBbkrwU11rzjvDeiorSnNmmfh59myylhQjO/qqcVMlNl+6YSuYidfvTk3K9FuAKNsCamVNdUeBK4i27LBVX5KjDrcu79LyK7xrkDVX1ZVWR2aQnfUzpEaows4mhhJQLOsqdOtWDVOwMD7UntrOnnJ9b7YAJlRTykWZtEuepN9hQvrtpNz6fsyaOj2stG186VgLU4DazHnKmsBP7A3EWBrB9QukeFJKdTH8u+AT1KYlXAZrdf720zHlO6VLmHom5FQx6h9OVJaNyXOQOh+2he4l08pO/bflvuc+XjLfQa/UJA8Oi2PyjHj9O+RwCzA21KQ462e+IG2lQLoB5olQJsCVydsFMF8qo194fYlmPImK1KhP4oh4ngDSwmyXZI0hX7H8hcNNTLZ/evr1+dWSdVEcKDYdtr1WGOtDpXh97CguM5+rV8k2/1ZXCW4g3R3Q5KmrGpDVhWhvonzLI2NvaWhctvKbr/89szjwLAtwwxM/1eJc6iqWyVXALLCgfoL8CqRBWZmqICP/5L+gZ6mbqJQgq4xhQLTC4DvKiJzf1BPmLxTj2wrOWXg9lbjzjwC1+BtE8A0IXhnrCK70tjlpZ+LeAmqqSkpsbwGBPLt+5+9C0oejw/UkyODCf22zyqSUr0mdmSg7x2poBeMrPo60zAMwuCEScV8FxGOUaMuTP2hn8lf9LkU0vO1BcEZcJmdwuwe9uRPWZyn5Wu/7RUK2welBmIPlqbbHVSQ0V5W/vgE35QNau5nBYZ79CDAgpsk9yqfteYorPZFVF8DC+LuBtpDDKCwLXRG4duy2gDP3CrYZEtUc7XbXfcwd445YSTJaaj7ECFCPlKhUDEeMFXiZFq1ju9oqwRdAylwjZgl9wzUzKOhd6G81EBeJ5iuwWp5zCx1MhAHXpUldUFF/L7tM2jROWTFL+mxIXRFqZJHhMb14CAdNGpQ+hdEbeU/qFAeowcVkZIH5bMShaKe8Bes+ItHv92Ciiy8yDnRTntP+DLKcpZLgDZFGlymu1AYFnJkHzlLqIenxKTle+D7goYCrelWsv5tBoxAswnHOj4DVJNQAKPXNaXsgUelZmafjKRtFztADfAN4Ad5HUdkj7gZgAM9p4MFQ2fewGytkdz9sCN7KVBqZCUpqOSWC4xjcj1SZqPvO6oY1jO7WkDIObjjI9uOZc6nuqx28lnINRvgqYDW9czfNf4pNGQtcGKTrBCaCLs3t5ZaAcfxsz2+L79kEwp0tpBUzlKsy6CDk4m7IxwEiO86wwTYNNvOzl7qAJgaS1U0TZZxA4sYVjLWeuhhLwxkcmAyxjmEjcz8tEMslAgW6qRWncREyR/DRPUm29H04yzOeWYjhoZU2yK0L908rOJb/1NxoAgsCN5iOQHirYMhD/KTBjW0IWBmefcRGK+bmneI9aRBMFKxTmI88jTGm/2ZdJipnRwcFJq5vX89P0VJ2Zmq3WF8Vi4ys/IsW8NskfMLJhmwJc4KljT1foecApOibvIZ+skIX7Le1HvQnLDHNsLAlw7aMkUnn7pUCT2rz46tJvtdioSQQ33QI3HaAGBtoT3rK29f9e2gqvpo1NS0G1TVno26Op0iVH9Tg3j9LeRZkpSujVwHS8Z6s8AAqi8c+DD0l/0thg5Dbxag26q/6rrN7TJCdt5KJso2xg+hV5bFwGR67pM83tXoPLPYor9Qpvw4amMpMAC3I/4X25qwu550AqCrSmep6N+NALMrwc/GiSIDq6HBQAEXLDPtwZmagYuu0DMljuc0tOksCrQuTjgoygBYg6JCa9O7tmYOOCoAurI4tGIdtBZ1rjX5noqx1WHbaQkOr/J69qWEaqr3OarkgzGEThxh5lzItXk67cJ3xvvthrDb6rqZ7TYw2DlU6KyyXNuSFOs8q68jmvqZVBmNjXV65NijW368KctjMPckNeRT7PdUD7rUsKtpTSPPtKCGHtiYtT/bYioPghSCRf1QUSqMxEJaH3yGCDzV9O3Fq/MLdLNEO+pbQAma70cj3oRADJlYfpYJq0wYq6zP8vDHyMPyGHn4+wnykHyWB6s8dBZ/nyXiFIlAh/gYeUiOkYe3J8gD+SwPVnkwlrGfpeGPsQ/kGHm4PFIekDt4riNS5KQNOplY6dqpU5NWLMXPuTP3UffdRHJoGsmtNXe+bqgnnerr5LCTzGNkgA3ekju0U2bto39XzLVRI2LvziC8iJAeDpX+m3d5t/O+usAhkoIdrs7vvZu4ykhNd/mhZ0LjYnQu0BHoUxkApJdSpGzuNZAwy3MFNM8B8GK5Rey9fONt/vE7hTXxrmDUGAaoSXXDe9tm9RZjip4UBA8TvkmceuVKAV3DMIoGRtrKjoeZ7ZigQOHgiL6g0IGnsWfapRFRtFPLnn2nh/2OUPGOuJ2q2w2+dml0JztpytCyL9sTonbZAFd8TdP4H08hW/e1aSQ2erCh4PH6d0NpDwmx2pWDP+3VHGao0db2AJWUzBJbYI7BcJYHtuCvKBQmyBcSRVJfnChwQxTBROu4mfH1BTSz9/0yatN7sOevLnx7OLMvnpbw3QS6IQNaqBYC9bSclD+PKYtvo0+1kFKNokO1fGql4Fyz6AF1pgxAB0h98AWd9xePUQtBOYD7/xmlPXp2MFJVGBytUhx78uXoeViHFCsj6mxpHcL4sH0t361DXXrTvrXVyuTcVmHhbs/wNtWY8KkwtF2qXkBKTRVe366XCo+ttty1VTwhSm7prVstclh7fGvWTKw1E6MmGwuD2h3XrbV24qhtwrZYo7/7T0nWPLwPZUv3CxU5mXPZWKgvD+1Mqc2ftjvVtlQsqrjvR4qWofzsRpUmyvP0+KhWrCrb7seTaMrLo8zjTFxFJ28kEj3UbVqc02w3cb2RcERQYimW6yLZQpAYZrAsS8z3YLnqp6Zz9uQZpmaeoZLb4U7pcGYTah5WX/ao7ovJw0ibW/Dj6tmDjwgDTsc/j6Fx8YFKa7q00F8ZsFsWoHXbR6NehwgzpMG48zpQz/zwcxIiyHQgaVLNlWRnO+Qxea4U6EtYBM8XwYGzy8y35Sd00j07b965qQLP4MDq3DzOImeDd4JgGT/uLHWMdpZGGYWsYZ56xlM4spCe+X1F31NJoOegi7ZX41SFGLw48iNTDljt+Uu8nsAz354vgt7UFr6/PvR2MdcRticsynQ25Ia8lzf9lht5dSd7+h1G9JmW6OVQt8tgMO3zpMMYQDvrPbJG4wY6fU9NnDy61RnyFs2lqC4YRMtIKXYTrDSW3MPisq0kuLJclDFyqtBbUvhwDszgeVNkNNvRf7iivfkWtaoM96KBddyQO9aghybB2wR31YhYx0B9CP1SD1r50K0eVNPcNcObQmVfWUd07gSRAxG7LLabQBxtxUA6iGhhfBMDywSunBvZOUU6Jv7JN8GwPB1sI4ZxOK6AJ8orXFYAL8ndpyf6L6a3NHiR3x4KPnAKhHqG7bNlE8p90NiyCXXI/TNXsy3McORwEL5uSxRX4D+O2WexkaRnx8VlJ/R9mHd+xyZ092SMKoBiQm8HfivGI3XJGYawb2IMNnu3FVrUyttjyHtJkhh+eLTPye7eK7dZw2LYwkvheciTQ1EwY0dIvdzQJJj0ik9zvS3XLHeOGlo5XK2pg2yXkRKC3yMR3RTtoc6365JF1Q3Xb9CUqrkzx+4tTN0qs8PHgAMNLyzJWx4DYY9j5WRnB4A8wslaiFOck30h7vniqdiWfHQaPGKK15pkPnwVMjipXLmovZNOzrl3zEYqTjFmaLhDA0OBCRTPdM8EOcdKL8FIDdbyXRrHFVS8eXv/1OywJ+rwdQdcPRp6nQsAhjm3DC/0QqSpZ/EYBaqk/GPUSxW85/WkeJDtxtwwamNtevBS6LHBTo96uf9QB0b9IsBkB/qkZVW+JzncF4IWhi/U/dyAIaOS+8LO6PEjBE1O2O5UYJhS+qED9E/kBw7GjGkm9EsN40BjmlCfTKjuewWOzGqNbA/ds9ocHWxWzGGienpsziMInY3bC4x4W22YRsPHUffsvz5kBTVtlQO53ZaUbpeHxy+W/UjmUlxby+0fIalzr4jKaLsfRSXQvvF0w+gflTekqrLUfl56mP3svcq2pneXkPRJVta8gVc1ruvuWSm0rdCFQ/T5AOjVBfhDBBrajVUVadKkaG81E95/gN7SGmDy6yPVt4ZNlXP+I2yqedDUYj+tVpBtbpi8hPswfVc0HraOx5lH5Ybnj2cZCQ2b4B+bzVQsFyO5hR0C1xLBDsHOpydYz+eMfQy2wCfaTKdtE+ebBfz5+eJkQze3WCGx5Nft0Mc1N3ZVYueHXvtEb5UGp327az6SjVJ6GGKn0pJrBWNvRLdNOH/ZQxE32Q3x/zijdJyVcR37Na+Mt6gz/jGgRf/ia47/Lz7p8guvN6P3Zfee55U0nslffECMlNGgBZVpMF69xm9gBZJTUgMM+0YWS6dW+Qg3MVpNex1XKT3nG718s/FZbKs7qqOMkKBa2B3ep7FDEttiS6FFvxl4UmasGI0uIgbaGmO6llZabpj+4v+ofdLERVgmeYF6YOGSP6OlOmB4CL0Os9fw3JQZ0JpVVTfen8UsIdDPZumZzNJGN0srjXTdkOZRdqt7gYIDTKj1PlMfPpvEfxWTuPk0JlElPtTQeOGz8fxsPP+UxpPcoWmgQ2yef8WmQvf7DbEePzzZitZTmXOEl6Mu3Bve5hX6x2yN21WscwbKNz+EADmCqdTq08uWddP/aWy/4NKpctuvOlc5SW5QNZGXUUnLbDGCdz45b1WAIyugqwKG3DNpjJDHES0mEEbRt31Gr+lqmaR/YHjUjp3FkubV8q2Gpw9Qg2r1XPoctF4n7VkdtUHO2kkOW3fqz++4DXbennxP2TM4cYYcHLBeDk47wunrE263vR9Jzcm+mYJt6OctWh0daE5j3c0HmjPfMlTnsFBcOnETvaEc9cRWtpdSqEMxU+ZMX1OcGnmMHuSxu37PE2VLOVf79F3NA97n0zzQfi/0aZ7oAG+061D2uKISU44W+vk+Y9nAj/jJbwXxtQc/usfvo+MrZguKLW6rwbiWZtwNK/dVom03K1rdcxoKV2apbU95iCt8lDscPUikASvrznGvg6waN0E1uwdskZjAyJg/0ice6BcP8I2tUq16yopiEenlTke6VjzpqtwXKX42kZ7Ao180ZYYadXMMFsK84MByfvNjuKTDbiw/1sNvZ43z7AtRcdk5NByHM2jEe0xJ68pH5zJM/xIsNanUhmWldbPd1423pEf/b7KU3+tPJ3iGM6S3ZnJq4lfZ/cCWYPpUF7B7j4Lauv+TPt1bDKm3am3Rc02o32kgv2xwe00KkxQSPewMY34vv0liI7H7On/bAez+qwlwbrYu/jKzX/tho8+hizRsHYSe+2YLGw1ttUcnX66gK5Kjz+r2nRUdeGB3+LUGT7+nQFcodL5ooB1z/3i2WYfaCaXZuGTQRQBRBN51FKF3OpKHPvVPWnO3xe/ev2mWLJ0libOkJyXd795epZY4+FWt4rrqpzs81+Fpvz+pXxuPNYtcrdHJmVMLrQkLlgr6xlKngiW85dv9AVF6KNka6i3+F5K6nB2iiwAA',
    'hw14_scripts/verification_scripts/verify_group_c.py': 'H4sIAOeP0GkC/6VWbW/bNhD+rl9x4CcJS9w4Wxs1gAekrjdk6Boj9ooCWUDQ0tkmoreRtJ0gyH/fkZIl0/aCLdMninfPc8eHdyTnqsyhEmaZyRnIvCqVgTH9Bs040esgCMa3N7+NhlN+e3MzhYFzCDmfyww5j3oKdZmtMYx6lVBYGH13fh+Mvo9b9130O2DLTf8njo8VKplbd2YnF6pcVTzhM6EeeKJKrVnw9dOrFDpRsmrgRWlwVpYPhAqCFOeQC1mEEZz+DLIwlwHQJ7RGWlK4pT0alafqiatV0ZPVUzFjUQ8fpTY6jP4DxXyVZcfxlaJsQjYdTaZwBuOryeQSfrUEMIR2CeAgPRYFDjMTGjNZIKe9IClaYSk2idjnW3sthD/VIwxzLBtplh5Vr6ywCAvc2IkBYyeARVKmslgM2MrMT2MW0XJhKYo0w1pALxtVbjSlk1GqoaX7LBNziyJFFdaYyJMso2AeNoLBAM5PYM5GVAqJwRTOW3awHmQrV0UKz4fYF3YgZ39PzpZqOPkGUkNS5lWGBltdSamL45peuM1k3t+eklvwG1V08Dco2OJq9c58+c6AxnBxKF4Ha4TTqxlPyG70JaQU9k4bdWI75Z4Sen5xPvNSWSaa7dLtVmAZuh4mFJnvmD/L7j33OuCd72MDdsbeAk3oO5zAWQQ/QH9Xij0Au7hiTpAPQLodWj+9ah3W1thq+UeBWzVrKbciPXewrYRijTwvU7R7+Eyrr+nctBKmtlE9MKI/JiXIOXQoXzeXkM3sxVv1TkQKyZZCpVRrmPL+hwcbydoyXggj18he9tZj0ac2MXAUdkkt35F2Om/aqa2oRopaQ58L6Ngns1IUabe94uPtFTfHpDHtkbUzddho8f9rtPiNjRZ3jfbR67OPTpT4aJvFfps9aYP5tkCoPdw/lym736uJGldvt8FH02HsHzdi8SpkWyHbeLY8dIWYLM17Vxl5LTcN3T1V0Z2N5rBGarytjXrUrKOhbxJztYdZVvK0XFjKLBO5aMiLB9oSrvCvFerDAJYBKDJFcGQ+v8iysG0JoRXHtchWwpTqoIcanSnk71JrCgmev/XqNumfgyi7edzQO4Jv8N9E8QHHwuw20Y87TRS7G0goqcuChmtUYoFHL6aG4c9iMh2N4T18G91e/3I9vJpe33y9hKsvX8CSTxz36DOra1ehWakCzuj5QwcLp1Mgp8eZO0c4t48hzlndFkpIjTBxGzx6lCasn0rR39zx9uILCgAA',
}

for rel_path, encoded_payload in SYNC_PAYLOADS.items():
    target_path = PROJECT_ROOT / rel_path
    target_path.parent.mkdir(parents=True, exist_ok=True)
    target_path.write_bytes(gzip.decompress(base64.b64decode(encoded_payload)))
    print(f'Synchronized {rel_path}')

Synchronized hw14_scripts/kamp_hw14.py
Synchronized hw14_scripts/hw14_experiment_runner.py
Synchronized hw14_scripts/verification_scripts/verify_group_c.py


In [23]:
runner_path = PROJECT_ROOT / 'hw14_scripts/hw14_experiment_runner.py'
runner_source = runner_path.read_text(encoding='utf-8')

if '_get_cmu_arctic_xvector_manifest' in runner_source:
    print('SpeechT5 speaker embedding hotfix already present in Drive copy')
else:
    old_block = '''def load_reference_speaker_embedding(embedding_id: int = 7440) -> np.ndarray:
    try:
        from datasets import load_dataset

        try:
            dataset = load_dataset(
                "Matthijs/cmu-arctic-xvectors",
                split="validation",
                trust_remote_code=True,
            )
        except TypeError:
            dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
        return np.asarray(dataset[int(embedding_id)]["xvector"], dtype=np.float32)
    except Exception:
        try:
            from genaibook.core import get_speaker_embeddings

            return np.asarray(get_speaker_embeddings(int(embedding_id)), dtype=np.float32)
        except Exception as exc:
            raise RuntimeError(
                "Unable to load the CMU Arctic speaker embedding required for SpeechT5 experiments."
            ) from exc
'''
    new_block = '''@lru_cache(maxsize=1)
def _get_cmu_arctic_xvector_manifest() -> tuple[Path, tuple[str, ...]]:
    from huggingface_hub import hf_hub_download
    import zipfile

    zip_path = Path(
        hf_hub_download(
            repo_id="Matthijs/cmu-arctic-xvectors",
            filename="spkrec-xvect.zip",
            repo_type="dataset",
        )
    )
    with zipfile.ZipFile(zip_path) as archive:
        entries = tuple(sorted(name for name in archive.namelist() if name.endswith(".npy")))
    if not entries:
        raise RuntimeError("No CMU Arctic xvector files were found in the downloaded archive.")
    return zip_path, entries


def load_reference_speaker_embedding(embedding_id: int = 7440) -> np.ndarray:
    try:
        from io import BytesIO
        import zipfile

        zip_path, entries = _get_cmu_arctic_xvector_manifest()
        index = int(embedding_id)
        if index < 0 or index >= len(entries):
            raise IndexError(f"embedding_id {index} is out of range for {len(entries)} CMU Arctic xvectors")
        with zipfile.ZipFile(zip_path) as archive:
            with archive.open(entries[index]) as handle:
                return np.asarray(np.load(BytesIO(handle.read())), dtype=np.float32)
    except Exception:
        try:
            from datasets import load_dataset

            try:
                dataset = load_dataset(
                    "Matthijs/cmu-arctic-xvectors",
                    split="validation",
                    trust_remote_code=True,
                )
            except TypeError:
                dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
            return np.asarray(dataset[int(embedding_id)]["xvector"], dtype=np.float32)
        except Exception:
            try:
                from genaibook.core import get_speaker_embeddings

                return np.asarray(get_speaker_embeddings(int(embedding_id)), dtype=np.float32)
            except Exception as exc:
                raise RuntimeError(
                    "Unable to load the CMU Arctic speaker embedding required for SpeechT5 experiments."
                ) from exc
'''
    if old_block not in runner_source:
        raise RuntimeError('Expected old SpeechT5 speaker embedding block was not found in Drive copy')
    runner_path.write_text(runner_source.replace(old_block, new_block, 1), encoding='utf-8')
    print('Applied SpeechT5 speaker embedding hotfix to Drive copy')

kamp_path = PROJECT_ROOT / 'hw14_scripts/kamp_hw14.py'
kamp_source = kamp_path.read_text(encoding='utf-8')
old_best_path = '    best_evaluator_reference_path = directories["full"] / "best_asr_evaluator_reference.json"\n'
new_best_path = '    best_evaluator_reference_path = directories["root"] / "best_asr_evaluator_reference.json"\n'
old_summary_path = '    summary_path = directories["full"] / f"group_c_{mode}_summary.json"\n'
new_summary_path = '    summary_path = directories["root"] / f"group_c_{mode}_summary.json"\n'

if new_best_path in kamp_source and new_summary_path in kamp_source:
    print('Group C summary path hotfix already present in Drive copy')
else:
    if old_best_path not in kamp_source or old_summary_path not in kamp_source:
        raise RuntimeError('Expected Group C summary path lines were not found in Drive copy')
    kamp_source = kamp_source.replace(old_best_path, new_best_path, 1)
    kamp_source = kamp_source.replace(old_summary_path, new_summary_path, 1)
    kamp_path.write_text(kamp_source, encoding='utf-8')
    print('Applied Group C summary path hotfix to Drive copy')

SpeechT5 speaker embedding hotfix already present in Drive copy
Applied Group C summary path hotfix to Drive copy


In [24]:
from contextlib import redirect_stderr, redirect_stdout

import importlib

import hw14_experiment_runner
hw14_experiment_runner = importlib.reload(hw14_experiment_runner)

import kamp_hw14
kamp_hw14 = importlib.reload(kamp_hw14)

from kamp_hw14 import run_group_c_bark_cross
from hw14_data_utils import CHECKPOINT_PATH, TeeLogger, load_checkpoint, save_checkpoint, save_text_artifact

print(f'hw14_experiment_runner imported from: {hw14_experiment_runner.__file__}')
print(f'kamp_hw14 imported from: {kamp_hw14.__file__}')
print(f'Checkpoint path: {CHECKPOINT_PATH}')

hw14_experiment_runner imported from: /content/drive/MyDrive/kamp_hw14/hw14_scripts/hw14_experiment_runner.py
kamp_hw14 imported from: /content/drive/MyDrive/kamp_hw14/hw14_scripts/kamp_hw14.py
Checkpoint path: /content/drive/MyDrive/kamp_hw14/hw14_experiments/checkpoint.json


In [5]:
DRY_RUN_ROOT = PROJECT_ROOT / 'hw14_experiments' / 'group_c_bark_cross' / 'dry_run'
DRY_RUN_LOG = PROJECT_ROOT / 'hw14_printouts' / 'group_c_bark_cross_dry_run_log.txt'
DRY_RUN_SUMMARY_PATH = DRY_RUN_ROOT / 'dry_run_summary.json'
DRY_RUN_CHECKPOINT_SNAPSHOT = DRY_RUN_ROOT / 'dry_run_checkpoint.json'

DRY_RUN_ROOT.mkdir(parents=True, exist_ok=True)
DRY_RUN_LOG.parent.mkdir(parents=True, exist_ok=True)

print(f'DRY_RUN_ROOT = {DRY_RUN_ROOT}')
print(f'DRY_RUN_LOG = {DRY_RUN_LOG}')

DRY_RUN_ROOT = /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_c_bark_cross/dry_run
DRY_RUN_LOG = /content/drive/MyDrive/kamp_hw14/hw14_printouts/group_c_bark_cross_dry_run_log.txt


In [20]:
summary = None

with TeeLogger(DRY_RUN_LOG) as tee, redirect_stdout(tee), redirect_stderr(tee):
    print('[NOTEBOOK] Starting Group C dry run')
    summary = run_group_c_bark_cross(mode='dry_run')
    print(summary)

save_text_artifact(DRY_RUN_SUMMARY_PATH, summary, as_json=True)
print(summary)

[NOTEBOOK] Starting Group C dry run
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
[timer] ga20h_baseline: 41.001s
[audio] Saved ga20h_baseline -> /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_c_bark_cross/dry_run/exp1_baselines/ga20h_baseline/ga20h_baseline.wav @ 16000 Hz
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
[timer] ga20k_baseline: 24.261s
[audio] Saved ga20k_baseline -> /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_c_bark_cross/dry_run/exp1_baselines/ga20k_baseline/ga20k_baseline.wav @ 16000 Hz
The attention mask and the pad token id were not set. As a c

In [21]:
current_checkpoint = load_checkpoint(CHECKPOINT_PATH)
save_checkpoint(current_checkpoint, DRY_RUN_CHECKPOINT_SNAPSHOT)
print(f'Saved dry-run checkpoint snapshot: {DRY_RUN_CHECKPOINT_SNAPSHOT}')

Saved dry-run checkpoint snapshot: /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_c_bark_cross/dry_run/dry_run_checkpoint.json


In [22]:
assert summary['counts']['baseline_rows'] == 2, summary['counts']
assert summary['counts']['exp7_rows'] == 4, summary['counts']
assert summary['counts']['exp8_rows'] == 1, summary['counts']

assert (DRY_RUN_ROOT / 'exp1_baselines' / 'exp1_baselines.csv').exists()
assert (DRY_RUN_ROOT / 'exp7_bark' / 'exp7_bark.csv').exists()
assert (DRY_RUN_ROOT / 'exp8_cross_tts' / 'exp8_cross_tts.csv').exists()
assert DRY_RUN_LOG.exists()
assert DRY_RUN_SUMMARY_PATH.exists()
assert DRY_RUN_CHECKPOINT_SNAPSHOT.exists()
print('Group C dry-run smoke test completed successfully.')

Group C dry-run smoke test completed successfully.
